In [20]:
from pymongo import MongoClient, UpdateOne
from pymongo.errors import OperationFailure
from tqdm.auto import tqdm
from datetime import datetime
from itertools import islice
import numpy as np
import math

In [21]:
HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

client = MongoClient(HOST, PORT)
db = client[DB_NAME]

group_checkin_col = db["0. US_group_checkin_labeled"]
user_tl_emb_col = db["user_tl_pattern_embeddings_attn_v1"]
user_cluster_vec_col = db["4. user_cluster_embedding_bestk"]
group_emb_col = db["group_embeddings"]

print("group_checkin count:", group_checkin_col.estimated_document_count())
print("user_tl_emb count:", user_tl_emb_col.estimated_document_count())
print("user_cluster_vec count:", user_cluster_vec_col.estimated_document_count())

group_checkin count: 184256
user_tl_emb count: 43954
user_cluster_vec count: 67207


In [22]:
def l2_normalize(vec: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norm = np.linalg.norm(vec)
    if norm < eps:
        return vec
    return vec / norm

def softmax(x):
    x = np.array(x, dtype=np.float32)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)




In [23]:
def batched(iterable, batch_size):
    iterator = iter(iterable)
    while True:
        batch = list(islice(iterator, batch_size))
        if not batch:
            break
        yield batch

def load_user_tl_embedding_cache(user_tl_keys, query_batch_size=500):
    cache = {}
    key_list = list(user_tl_keys)

    for key_batch in batched(key_list, query_batch_size):
        query = {
            "$or": [
                {"user_id": user_id, "TL": tl}
                for user_id, tl in key_batch
            ]
        }
        cursor = user_tl_emb_col.find(
            query,
            {"_id": 0, "user_id": 1, "TL": 1, "pattern_embedding": 1}
        )
        for doc in cursor:
            cache[(str(doc["user_id"]), int(doc["TL"]))] = np.array(
                doc["pattern_embedding"],
                dtype=np.float32,
            )

    return cache


In [24]:
def load_user_cluster_score_cache(user_ids):
    cache = {}
    user_id_list = list(user_ids)
    if not user_id_list:
        return cache

    cursor = user_cluster_vec_col.find(
        {"user_id": {"$in": user_id_list}},
        {"_id": 0, "user_id": 1, "embedding": 1}
    )
    for doc in cursor:
        vec = doc.get("embedding")
        if vec is None:
            continue
        cache[str(doc["user_id"])] = np.array(vec, dtype=np.float32)

    return cache


In [25]:
def build_group_embedding(users, TL, cluster_id_k, tl_embedding_cache, cluster_score_cache, beta=10.0):
    valid_user_ids = []
    user_embeddings = []
    raw_scores = []
    cid = int(cluster_id_k)

    for user_id in users:
        user_id = str(user_id)
        emb = tl_embedding_cache.get((user_id, int(TL)))
        score_vec = cluster_score_cache.get(user_id)

        if emb is None:
            continue
        if score_vec is None:
            continue
        if cid < 0 or cid >= len(score_vec):
            continue

        valid_user_ids.append(user_id)
        user_embeddings.append(emb)
        raw_scores.append(float(score_vec[cid]))

    if len(valid_user_ids) == 0:
        return None

    user_embeddings = np.stack(user_embeddings, axis=0)  # (n, 64)
    raw_scores = np.array(raw_scores, dtype=np.float32)

    # attention weights
    attn_weights = softmax(beta * raw_scores)  # (n,)

    # weighted sum
    group_emb = np.sum(user_embeddings * attn_weights[:, None], axis=0)
    group_emb = l2_normalize(group_emb)

    return {
        "valid_user_ids": valid_user_ids,
        "raw_scores": raw_scores.tolist(),
        "attention_weights": attn_weights.tolist(),
        "group_embedding": group_emb.tolist(),
        "n_valid_users": len(valid_user_ids),
    }

In [26]:
sample_group = group_checkin_col.find_one({})
sample_group

{'_id': ObjectId('699ed7cc0f6b516462485585'),
 'TL': 2,
 'VenueID': '4c143cada5eb76b0dc7dc1b7',
 'cluster_label': 3,
 'Latitude': 31.188807,
 'Longitude': -81.376461,
 'VenueCategoryname': 'American Restaurant',
 'Users': [1059003, 14732],
 'users': 2,
 'Group_ID': 1,
 'cluster_id_k': 2}

In [27]:
users = sample_group["Users"]
TL = int(sample_group["TL"])
cluster_id_k = int(sample_group["cluster_id_k"])

sample_user_tl_keys = {(str(user_id), TL) for user_id in users}
sample_user_ids = {str(user_id) for user_id in users}

sample_tl_embedding_cache = load_user_tl_embedding_cache(sample_user_tl_keys)
sample_cluster_score_cache = load_user_cluster_score_cache(sample_user_ids)

result = build_group_embedding(
    users,
    TL,
    cluster_id_k,
    sample_tl_embedding_cache,
    sample_cluster_score_cache,
    beta=10.0,
)
result

In [28]:
def ensure_index(collection, keys, **kwargs):
    try:
        return collection.create_index(keys, **kwargs)
    except OperationFailure as exc:
        if exc.code == 86:
            print(f"skip existing conflicting index on {collection.name}: {keys}")
            return None
        raise

ensure_index(user_tl_emb_col, [("user_id", 1), ("TL", 1)], unique=True)
ensure_index(user_cluster_vec_col, "user_id")
ensure_index(group_emb_col, "source_group_checkin_id", unique=True)
ensure_index(group_emb_col, [("Group_ID", 1), ("TL", 1)])

skip existing conflicting index on 4. user_cluster_embedding_bestk: user_id


'Group_ID_1_TL_1'

In [29]:
GROUP_BATCH_SIZE = 1000
BULK_WRITE_SIZE = 1000
BETA = 10.0

projection = {
    "_id": 1,
    "Group_ID": 1,
    "Users": 1,
    "TL": 1,
    "cluster_id_k": 1,
    "Latitude": 1,
    "Longitude": 1,
    "VenueID": 1,
    "VenueCategoryname": 1,
    "users": 1,
}

written = 0
processed = 0
skipped = 0
cursor = group_checkin_col.find({}, projection, batch_size=GROUP_BATCH_SIZE)
total_docs = group_checkin_col.estimated_document_count()

for doc_batch in tqdm(
    batched(cursor, GROUP_BATCH_SIZE),
    total=math.ceil(total_docs / GROUP_BATCH_SIZE),
    desc="Building group embeddings (batch)",
):
    user_tl_keys = {
        (str(user_id), int(doc["TL"]))
        for doc in doc_batch
        for user_id in doc["Users"]
    }
    user_ids = {
        str(user_id)
        for doc in doc_batch
        for user_id in doc["Users"]
    }

    tl_embedding_cache = load_user_tl_embedding_cache(user_tl_keys)
    cluster_score_cache = load_user_cluster_score_cache(user_ids)
    ops = []

    for doc in doc_batch:
        processed += 1

        source_id = str(doc["_id"])
        group_id = int(doc["Group_ID"])
        TL = int(doc["TL"])
        cluster_id_k = int(doc["cluster_id_k"])
        users = doc["Users"]

        result = build_group_embedding(
            users,
            TL,
            cluster_id_k,
            tl_embedding_cache,
            cluster_score_cache,
            beta=BETA,
        )

        if result is None:
            skipped += 1
            continue

        out_doc = {
            "_id": source_id,
            "source_group_checkin_id": source_id,
            "Group_ID": group_id,
            "TL": TL,
            "cluster_id_k": cluster_id_k,
            "Latitude": float(doc["Latitude"]),
            "Longitude": float(doc["Longitude"]),
            "VenueID": doc.get("VenueID"),
            "VenueCategoryname": doc.get("VenueCategoryname"),
            "group_size": int(doc.get("users", len(users))),
            "input_user_ids": [str(u) for u in users],
            "valid_user_ids": result["valid_user_ids"],
            "n_valid_users": result["n_valid_users"],
            "raw_scores": result["raw_scores"],
            "attention_weights": result["attention_weights"],
            "group_embedding": result["group_embedding"],
            "dim": 64,
            "beta": BETA,
            "value_source": "user_tl_pattern_embeddings_attn_v1",
            "score_source": "4. user_cluster_embedding_bestk",
            "updated_at": datetime.utcnow(),
        }

        ops.append(UpdateOne(
            {"_id": source_id},
            {"$set": out_doc},
            upsert=True,
        ))

        if len(ops) >= BULK_WRITE_SIZE:
            group_emb_col.bulk_write(ops, ordered=False)
            written += len(ops)
            ops = []

    if ops:
        group_emb_col.bulk_write(ops, ordered=False)
        written += len(ops)

    print(
        f"processed={processed:,} written={written:,} skipped={skipped:,} "
        f"cached_tl={len(tl_embedding_cache):,} cached_users={len(cluster_score_cache):,}"
    )

print("done. written:", written)
print("done. skipped:", skipped)
print("group_embeddings count:", group_emb_col.estimated_document_count())

Building group embeddings (batch):   0%|          | 0/185 [00:00<?, ?it/s]

processed=1,000 written=410 skipped=590 cached_tl=1,687 cached_users=2,719
processed=2,000 written=897 skipped=1,103 cached_tl=1,703 cached_users=3,022
processed=3,000 written=1,348 skipped=1,652 cached_tl=1,142 cached_users=2,657
processed=4,000 written=1,758 skipped=2,242 cached_tl=1,612 cached_users=2,608
processed=5,000 written=2,236 skipped=2,764 cached_tl=1,924 cached_users=2,782
processed=6,000 written=2,644 skipped=3,356 cached_tl=1,191 cached_users=2,171
processed=7,000 written=3,071 skipped=3,929 cached_tl=1,129 cached_users=1,997
processed=8,000 written=3,552 skipped=4,448 cached_tl=1,221 cached_users=2,201
processed=9,000 written=4,002 skipped=4,998 cached_tl=1,278 cached_users=2,248
processed=10,000 written=4,383 skipped=5,617 cached_tl=1,097 cached_users=2,110
processed=11,000 written=4,793 skipped=6,207 cached_tl=1,046 cached_users=2,446
processed=12,000 written=5,218 skipped=6,782 cached_tl=905 cached_users=2,269
processed=13,000 written=5,637 skipped=7,363 cached_tl=1,

In [30]:
sample_out = group_emb_col.find_one({})
sample_out.keys()

dict_keys(['_id', 'Group_ID', 'Latitude', 'Longitude', 'TL', 'VenueCategoryname', 'VenueID', 'attention_weights', 'beta', 'cluster_id_k', 'dim', 'group_embedding', 'group_size', 'input_user_ids', 'n_valid_users', 'raw_scores', 'score_source', 'source_group_checkin_id', 'updated_at', 'valid_user_ids', 'value_source'])

In [31]:
print("Group_ID:", sample_out["Group_ID"])
print("TL:", sample_out["TL"])
print("cluster_id_k:", sample_out["cluster_id_k"])
print("valid_user_ids:", sample_out["valid_user_ids"])
print("raw_scores:", sample_out["raw_scores"])
print("attention_weights:", sample_out["attention_weights"])
print("embedding dim:", len(sample_out["group_embedding"]))

Group_ID: 7
TL: 2
cluster_id_k: 2
valid_user_ids: ['74385', '1859716', '118174']
raw_scores: [0.37703806161880493, 0.36368343234062195, 0.3791622817516327]
attention_weights: [0.3452494144439697, 0.30208876729011536, 0.35266178846359253]
embedding dim: 64
